In [ ]:
import os, re, json, requests
import pandas as pd
from pathlib import Path

BASE = Path(r"D:\研\大论文\数据\评论数据（原数据＋预处理后数据）\预处理后数据")
GOV = r"D:\研\大论文\数据\热度数据\指标5-政府回应内容及前后2小时发文量.xlsx"
FISHER = r"D:\研\大论文\数据\热度数据\分年份数据\Fisher分段结果汇总.xlsx"
CLASS = r"D:\研\大论文\数据\热度数据\SPSS聚类\4分类.xlsx"
OUT = r"D:\研\大论文\数据\热度数据\三个主体参与度与发言侧重.xlsx"

KEY = os.getenv("DEEPSEEK_API_KEY")
STAGES = ["爆发期", "成熟期", "衰退期"]
KOL = ["意见领袖", "名人", "达人", "专家", "大V"]

FUN = {
    "政府": ["信息披露", "风险沟通", "公众沟通", "调查进展", "下架召回", "处罚问责"],
    "媒体": ["信息传播与事实核实", "解释分析与风险提示", "舆论监督与责任追踪", "公众沟通与社会动员"],
    "意见领袖": ["信息传播与信源提供", "分析解释与专业解读", "示范引导与支持交流", "网络动员与行动倡议"]
}

col = lambda d, ns: next((x for x in ns if x in d.columns), None)
evname = lambda x: re.sub(r"\.csv$|_小时数据$", "", str(x).replace("微博源数据-", "")).strip()
year = lambda x: (re.search(r"20\d{2}", str(x)) or [None, ""])[1]

def csvread(p):
    for enc in ["utf-8-sig", "gb18030"]:
        try:
            return pd.read_csv(p, dtype=str, encoding=enc)
        except Exception:
            pass
    return pd.DataFrame()

def label_text(txt, labels):
    try:
        q = f"判断文本体现的功能，可多选；符合记1，不符合记0，信息不足记null。只返回JSON。标签：{labels}\n文本：{txt}"
        r = requests.post(
            "https://api.deepseek.com/chat/completions",
            headers={"Authorization": f"Bearer {KEY}"},
            json={"model": "deepseek-chat",
                  "messages": [{"role": "user", "content": q}],
                  "temperature": 0,
                  "response_format": {"type": "json_object"}},
            timeout=60
        )
        r.raise_for_status()
        d = json.loads(r.json()["choices"][0]["message"]["content"])
        return {x: d.get(x) if d.get(x) in [0, 1] else None for x in labels}
    except Exception:
        return {x: None for x in labels}


if not KEY:
    raise ValueError("请先设置环境变量 DEEPSEEK_API_KEY")

c, f = pd.read_excel(CLASS), pd.read_excel(FISHER)
g = pd.read_excel(GOV, sheet_name="逐条有效回应")

ec, yc, cc = col(c, ["事件名称", "事件"]), col(c, ["年份", "所属年份", "年"]), col(c, ["类别4", "分类4"])
ef, yf, sf, hf = col(f, ["事件名称", "事件"]), col(f, ["年份", "所属年份", "年"]), col(f, ["阶段类型", "阶段"]), col(f, ["小时序号", "小时"])
ge, gy, gt, gx = col(g, ["事件名称", "事件"]), col(g, ["年份", "所属年份", "年"]), col(g, ["回应时间", "发布时间", "时间"]), col(g, ["回应内容", "回应正文", "政府回应内容", "微博内容", "内容", "文本"])

if not all([ec, cc, ef, sf, hf, ge, gt, gx]):
    raise ValueError("请检查事件、时间、正文和Fisher阶段字段")

c["e"], c["y"] = c[ec].map(evname), c[yc].map(year) if yc else ""
f["e"], f["y"], f["h"] = f[ef].map(evname), f[yf].map(year) if yf else "", pd.to_numeric(f[hf], errors="coerce")

stages = {
    (y, e): sorted(
        [(str(s), int(x.h.min()), int(x.h.max()))
         for s, x in z.groupby(sf) if x.h.notna().any()],
        key=lambda v: v[1]
    )
    for (y, e), z in f.dropna(subset=["h"]).groupby(["y", "e"])
}

files = {
    (next((year(x) for x in p.parts if year(x)), ""), evname(p.stem)): p
    for p in BASE.rglob("*.csv")
}

meta, rows = {}, []

# 读取事件微博，提取媒体和意见领袖
for _, r in c.iterrows():
    e, y, cat = r.e, r.y, r[cc]
    file = files.get((y, e))
    if file is None:
        continue

    d = csvread(file)
    t = col(d, ["时间", "发布时间", "发文时间", "created_at"])
    ty = col(d, ["认证类型", "类型_final", "账号类型", "用户类型", "类型"])
    tx = col(d, ["微博内容_清洗后", "微博内容", "内容"])
    if not all([t, ty, tx]):
        continue

    d["t"] = pd.to_datetime(d[t], errors="coerce")
    d = d.dropna(subset=["t"])
    if d.empty:
        continue

    start = d.t.min().floor("h")
    d["h"] = ((d.t.dt.floor("h") - start).dt.total_seconds() // 3600).astype(int)
    rg = stages.get((y, e), [])
    den = {s: int(d.h.between(lo, hi).sum()) for s, lo, hi in rg}
    meta[y, e] = {"cat": cat, "total": len(d), "den": den, "start": start}

    for actor, mask in [
        ("媒体", d[ty].fillna("").str.contains("媒体", regex=False)),
        ("意见领袖", d[ty].fillna("").str.strip().isin(KOL))
    ]:
        for _, v in d[mask].iterrows():
            txt = str(v[tx]).strip()
            h = int(v.h)
            st = next((s for s, lo, hi in rg if lo <= h <= hi), "")
            rows.append([y, e, cat, actor, v.t, st, "" if txt.lower() == "nan" else txt])

# 读取政府有效回应
g["e"], g["y"], g["t"] = (
    g[ge].map(evname),
    g[gy].map(year) if gy else "",
    pd.to_datetime(g[gt], errors="coerce")
)

for _, v in g.dropna(subset=["t"]).drop_duplicates(["e", "y", "t"]).iterrows():
    k = (v.y, v.e)
    if k not in meta:
        continue
    txt = str(v[gx]).strip()
    h = int((v.t.floor("h") - meta[k]["start"]).total_seconds() // 3600)
    st = next((s for s, lo, hi in stages.get(k, []) if lo <= h <= hi), "")
    rows.append([v.y, v.e, meta[k]["cat"], "政府", v.t, st, "" if txt.lower() == "nan" else txt])

p = pd.DataFrame(rows, columns=["年份", "事件名称", "类别4", "主体", "发布时间", "所属阶段", "正文"])
p = p.drop_duplicates(["主体", "年份", "事件名称", "发布时间", "正文"])

# 识别三个主体的发言功能
for labels in FUN.values():
    for x in labels:
        p[x] = None

for i, v in p.iterrows():
    if str(v["正文"]).strip():
        z = label_text(v["正文"], FUN[v["主体"]])
        p.loc[i, list(z)] = list(z.values())
    if (i + 1) % 50 == 0:
        print("已识别", i + 1, "条")

# 参与度
part = []
for (y, e), m in meta.items():
    for actor in FUN:
        z = p[(p.年份 == y) & (p.事件名称 == e) & (p.主体 == actor)]
        n = len(z)
        a = {
            "年份": y, "事件名称": e, "类别4": m["cat"], "主体": actor,
            "主体发文数": n,
            "整体参与度": n / m["total"] if m["total"] else None
        }
        for s in STAGES:
            k = int((z.所属阶段 == s).sum())
            d = m["den"].get(s, 0)
            a[s + "发文数"] = k
            a[s + "参与度"] = k / d if d else None
        part.append(a)

# 发言侧重
focus = []
for (actor, cat), z in p.groupby(["主体", "类别4"]):
    for s in ["整体"] + STAGES:
        q = z if s == "整体" else z[z.所属阶段 == s]
        for x in FUN[actor]:
            v = q[x].dropna()
            focus.append({
                "主体": actor, "类别4": cat, "阶段": s, "功能": x,
                "功能发文数": int((v == 1).sum()),
                "有效识别数": len(v),
                "占比": (v == 1).mean() if len(v) else None
            })

with pd.ExcelWriter(OUT) as w:
    pd.DataFrame(part).to_excel(w, sheet_name="参与度", index=False)
    pd.DataFrame(focus).to_excel(w, sheet_name="发言侧重", index=False)

print("完成：", OUT)